# HKS crypts: reconstruction and alignment

Uses **20250929** and **20251201**, biological day **3.5 or later**. The HKS files provide membership masks only. Meshes are loaded in original coordinates; detector tips, distance axes and primitive fits are never used.

The pilot samples four crypts per dataset/timepoint group, stratified by mesh vertex count and favoring distinct organoids. This is a reproducible pilot, not validation of every crypt. Existing dataset blacklists are respected. Failures are reported and retained in the results.

Set `ORGANOGRAPH_DATA_ROOT` to relocate the data. Run using the `organoid-mesh` environment. This notebook writes validation reports/figures, not a training dataset.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/organograph").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display
pio.renderers.default = "plotly_mimetype"
from organograph.crypt_morphology import *
from organograph.plotting.crypt_morphology import (
    plot_cap_representation, plot_cap_alignment,
    plot_cap_error_summary, plot_cap_comparison_static,
)
OUT = ROOT / "exports/crypt_morphology"
OUT.mkdir(parents=True, exist_ok=True)
DEGREES = [4, 8, 12]
BASES = {degree: CapBasis.build(degree) for degree in DEGREES}

from organograph.crypt_morphology.datasets import discover_hks_crypts, select_validation_crypts, load_hks_crypt
DATA_ROOT = Path(os.environ.get("ORGANOGRAPH_DATA_ROOT", ROOT.parent / "NicoleData"))
DATASETS = ("20250929", "20251201")
MIN_DAY = 3.5
PER_GROUP = 4
SEED = 2026
records = discover_hks_crypts(DATA_ROOT, datasets=DATASETS, min_day=MIN_DAY)
selected = select_validation_crypts(records, per_group=PER_GROUP, seed=SEED)
(OUT / "selection.json").write_text(json.dumps(selected, indent=2))
print(f"Available eligible crypts: {len(records)}; pilot: {len(selected)}")
display(pd.DataFrame(selected)[["dataset", "timepoint", "label_uid", "crypt_index", "n_selected_vertices"]])

## Fit a common basis at several resolutions

All samples at a given degree use the exact same basis. Angular/area distortion, conditioning and bidirectional reconstruction errors are reported separately. Errors use original mesh coordinate units; relative errors divide by each original surface's RMS radius. A mapping failure does not trigger automatic hole filling or surface alteration.

In [ ]:
rows, fitted = [], {}
for rec in selected:
    key = (rec["dataset"], rec["label_uid"], rec["crypt_index"])
    identity = {k: rec[k] for k in ["dataset", "timepoint", "label_uid", "crypt_index", "n_selected_vertices"]}
    try:
        patch, metadata = load_hks_crypt(rec)
        mapping = parameterize_patch(patch)
        for degree, basis in BASES.items():
            rep = fit_representation(patch, mapping, basis, metadata=metadata)
            rv, rf = reconstruct(rep, basis, original_coordinates=True, n_rings=32, n_sectors=96)
            metrics = surface_distance_metrics(patch.vertices, patch.faces, rv, rf)
            rows.append(dict(identity, degree=degree, status="ok", **rep.diagnostics, **metrics))
            fitted[key, degree] = patch, mapping, rep
    except (ValueError, FileNotFoundError, RuntimeError) as exc:
        rows.append(dict(identity, degree=None, status="failed", reason=str(exc)))
results = pd.DataFrame(rows)
results.to_csv(OUT / "real_validation.csv", index=False)
display(results[results.status == "failed"])
success = results[results.status == "ok"]
if success.empty:
    raise RuntimeError("No valid reconstructions; inspect failure table")
display(success.groupby("degree")["relative_symmetric_rmse"].agg(["count", "median", "max"]))
figure = plot_cap_error_summary(success, group_column="timepoint")
figure.savefig(OUT / "real_errors.png", dpi=160)
display(figure)
plt.close(figure)

## Inspect the worst reconstructed crypt

By default this selects the largest geometric error at degree 12, so the inspection does not hide difficult cases. Set `INSPECT_KEY` explicitly to a `(dataset, label_uid, crypt_index)` tuple to inspect another sample.

In [ ]:
DEGREE = 12
worst = success[success.degree == DEGREE].sort_values("relative_symmetric_rmse").iloc[-1]
INSPECT_KEY = (worst.dataset, worst.label_uid, int(worst.crypt_index))
patch, mapping, rep = fitted[INSPECT_KEY, DEGREE]
print("Inspecting:", INSPECT_KEY)
display(pd.Series(rep.diagnostics))
display(plot_cap_representation(patch, mapping, rep, BASES[DEGREE]))
figure = plot_cap_comparison_static(patch, rep, BASES[DEGREE])
figure.savefig(OUT / "real_worst_comparison.png", dpi=160)
display(figure)
plt.close(figure)

## Compare physical-size and shape-only cohort alignment

Both modes use identical mappings and basis functions. Each alignment learns a separate template; their orientations need not coincide. The metric uses fixed physical area weights from the initial reference surface. Cohort similarity is not itself evidence of biologically correct correspondence.

The cap phase search is numerical and may have local optima. Convergence and phase ambiguity are printed. A frozen reference can later be applied to held-out crypts without refitting the cohort mean.

In [ ]:
basis = BASES[DEGREE]
keys = [key for key, degree in fitted if degree == DEGREE]
physical = [fitted[key, DEGREE][2] for key in keys]
shape_only = [with_size_mode(r, "unit_rms") for r in physical]
alignments = {}
alignment_rows = []
for mode, reps in [("physical", physical), ("unit_rms", shape_only)]:
    result = align_representations(reps, basis, max_iterations=200, tolerance=1e-6)
    alignments[mode] = result
    print(mode, "converged:", result.converged, "iterations:", len(result.objective_history),
          "objective:", result.objective_history[-1], "ambiguous phases:", int(result.phase_ambiguous.sum()))
    display(plot_cap_alignment(result, basis, labels=[str(k) for k in keys]))
    for iteration, objective in enumerate(result.objective_history):
        alignment_rows.append(dict(size_mode=mode, iteration=iteration, objective=objective,
                                   converged=result.converged))
pd.DataFrame(alignment_rows).to_csv(OUT / "alignment_validation.csv", index=False)
# One reference per size convention. These small artifacts are for reproducibility.
save_basis(OUT / "pilot_basis.json", basis)
for mode, result in alignments.items():
    reference = make_alignment_reference(result, basis)
    save_alignment_reference(OUT / f"pilot_alignment_{mode}.npz", reference)

## Interpretation before step 2

Check the opening geometry, the worst-case surface fit, chart distortion and the alignment overlays. Physical and normalized shape comparisons are both available. This stage deliberately retains the original hard HKS boundary: boundary perturbation, soft weighting and boundary optimization are subsequent experiments.

A successful reconstruction does not prove boundary independence, anatomical correspondence, or validity of arbitrary generated coefficients. The current distance diagnostics do not certify absence of self-intersections. The size-normalization radius currently depends on the selected patch; that sensitivity belongs in the boundary experiments.